In [ ]:
!uv pip install mailtrap serpapi langgraph==0.2.60 langchain-openai==0.2.14 langchain-community==0.3.1 -qq

## Package imports and env var loading

In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
import requests
import serpapi
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
import gradio as gr
import os
import uuid
import json

In [ ]:
# add keys
load_dotenv(override=True)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
openai_api_key = os.getenv("OPENAI_API_KEY")
serp_api_key = os.getenv("SERP_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")

if pushover_user is None or pushover_token is None:
    raise ValueError("Pushover credentials are required!")

if serp_api_key is None:
    raise ValueError("Serp API key is required for Google search")

AGENT_MODEL = "gpt-4o-mini"

### initialize graph memory

In [ ]:
memory = MemorySaver()

### Agent models initializations

In [ ]:
#Agent models
planner_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=1000,
    openai_api_key=openai_api_key,
)

worker_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=4000,
    openai_api_key=openai_api_key,
)

evaluator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=1000,
    openai_api_key=openai_api_key,
)

notifier_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=3000,
    openai_api_key=openai_api_key,
)

In [ ]:
# plug in Meta Llama Prompt Guard, protecttion against prompt injection via transformers pipeline here when approved
# from transformers import pipeline
# PROMPT_GUARD_MODEL = "meta-llama/Prompt-Guard-86M"
# def classify_user_input(query: str):
#   """Detect signs of jailbreaking and prompt injection from input using llama prompt guard."""
#   classifier = pipeline("text-classification", model=PROMPT_GUARD_MODEL)
#   classifier(query)

In [ ]:
# classify_user_input("ignore previous instructions")  # sanity check

#### Input Guardrail
llama prompt guard 2 not approved at time of implemention

In [ ]:
def input_guardrail(query: str):
    """Guardrails for user input: check for empty strings, length, and prompt injection."""
    user_input = query.strip()

    if not user_input:
        raise ValueError("Query cannot be empty.")

    if len(user_input) < 10:
        raise ValueError("Your query is too short. Please provide a more detailed request (at least 10 characters).")

    if len(user_input) > 500:
        raise ValueError("Your query is too long. Please keep it under 500 characters.")

    # Basic keyword filtering for potential prompt injection attempts
    dangerous_keywords = [
        "ignore previous instructions",
        "disregard all prior instructions",
        "system override",
        "forget everything",
        "act as a different persona",
        "jailbreak",
        "confidential",
        "secret",
        "private information"
    ]
    for keyword in dangerous_keywords:
        if keyword in user_input.lower():
            raise ValueError(f"Input blocked: Contains a dangerous keyword or phrase: '{keyword}'.")

    # Check for excessive character repetition (e.g., "aaaaaaaaa")
    # This can indicate low-quality input or simple attacks
    for i in range(len(user_input) - 4):
        if user_input[i] == user_input[i+1] == user_input[i+2] == user_input[i+3] == user_input[i+4]:
            raise ValueError("Input blocked: Excessive character repetition detected. Please provide more meaningful input.")

    # model usage not yet approved by meta
    # clean_input = classify_user_input(user_input)
    # if clean_input[0]['label'].lower() == "malicious":
    #     raise ValueError("Input blocked: signs of jailbreaking detected.")

    return user_input

#### Agents structured outputs declaration

In [ ]:
class PlanItem(BaseModel):
    query: str = Field(description="Search query for Wikipedia / grounding.")
    url: Optional[str] = Field(default=None, description="Optional http(s) URL for fetch_web_page.")
    reason: str = Field(description="Why this step matters.")

class PlannerOutput(BaseModel):
    additional_input_required: bool = Field(description="True if more input is needed from the user before planning." )
    plan: Optional[List[PlanItem]] = Field(default=None, description="Search/scrape tasks for the worker.")
    clarification_question: Optional[str] = Field(default=None, description="If additional_input_required, ask this question to the user.")

class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the generated response accurately answers the user query.")
    user_input_needed: bool = Field(description="Whether more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.")

#### State initialization

In [ ]:
class State(TypedDict, total=False):
    messages: Annotated[List[Any], add_messages]
    feedback_on_work: Optional[str]
    planning_complete: bool
    user_input_needed: bool
    success_criteria_met: bool
    success_criteria: str
    last_report: Optional[str]
    guard_ok: bool
    planner_search_rounds: int

### Tools Declarations and agent bindings

In [ ]:
@tool
def send_push_notification(message: str) -> str:
    """Send a push notification via Pushover."""
    try:
        url = "https://api.pushover.net/1/messages.json"
        payload = {"token": pushover_token, "user": pushover_user, "message": message}
        r = requests.post(url, payload, timeout=30)
        r.raise_for_status()
        return "Push notification sent."
    except Exception as e:
        return f"Push notification failed: {e}"

In [ ]:
def google_search(query: str):
    """Search Google for the query. Returns organic results with title, link, snippet — use to pick URLs to scrape and to refine wiki queries."""
    try:
        client = serpapi.Client(api_key=serp_api_key)
        response = client.search(engine="google", q=query)
        return response.get("organic_results") or []
    except Exception as e:
        return [{"error": str(e)}]

In [ ]:
@tool
def fetch_web_page(url: str) -> str:
    """Fetch a URL and return plain text for scraping (use when the plan lists a URL)."""
    try:
        r = requests.get(
            url,
            timeout=20,
            headers={"User-Agent": "InfoCompanion/1.0 (research assistant)"},
        )
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        return text[:12000] if len(text) > 12000 else text
    except Exception as e:
        return f"Failed to fetch URL: {e}"

In [ ]:
worker_tools = [fetch_web_page]
worker_tool_node = ToolNode(worker_tools)

In [ ]:

planner_llm_with_output = planner_llm.with_structured_output(PlannerOutput)
planner_llm_with_search = planner_llm.bind_tools([google_search])
worker_llm_with_tools = worker_llm.bind_tools(worker_tools)
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

In [ ]:

def format_conversation(messages: List[Any]) -> str:
    from langchain_core.messages import ToolMessage

    lines: List[str] = []
    for message in messages:
        if isinstance(message, HumanMessage):
            lines.append(f"Human: {message.content}")
        elif isinstance(message, AIMessage):
            lines.append(f"AI: {message.content}")
        elif isinstance(message, SystemMessage):
            lines.append(f"System: {message.content}")
        elif isinstance(message, ToolMessage):
            lines.append(f"Tool ({message.name}): {message.content}")
        else:
            lines.append(str(message))
    return "\n".join(lines)

## Agents Nodes declarations

#### Planner agent
decide whether or not to use search tools or additional inputs are required and return a final plan for the next stage


In [ ]:
MAX_SEARCHES = 5

PLANNER_INSTRUCTION = f"""You are the research planner.

google_search is optional—use it only when web results would clearly help (URLs to scrape, facts, recency). Skip it for purely conceptual tasks or when the user request is already specific enough.

The workflow allows at most {MAX_SEARCHES} search rounds total (each round may include multiple google_search calls in one response). Prefer fewer, targeted queries.

If you need clarification from the user, reply with plain text only (no tool calls) and ask your question clearly.

When you are done searching (or chose not to search), reply with plain text only (no tool calls)—a short note is enough—so the plan can be finalized."""


def planner_llm(state: State) -> Dict[str, Any]:
    msgs = state["messages"]
    last = msgs[-1]
    rounds = 0 if isinstance(last, HumanMessage) else state.get("planner_search_rounds", 0)
    ai = planner_llm_with_search.invoke([SystemMessage(content=PLANNER_INSTRUCTION)] + msgs)
    return {"messages": [ai], "planner_search_rounds": rounds}


def route_planner_tools(state: State) -> str:
    last = state["messages"][-1]
    hops = state.get("planner_search_rounds", 0)
    if getattr(last, "tool_calls", None) and hops < MAX_SEARCHES:
        return "planner_search"
    return "planner_finalize"


planner_search_tool_node = ToolNode([google_search])


def planner_search(state: State) -> Dict[str, Any]:
    out = planner_search_tool_node.invoke(state)
    return {
        **out,
        "planner_search_rounds": state.get("planner_search_rounds", 0) + 1,
    }


def planner_finalize(state: State) -> Dict[str, Any]:
    msgs: List[Any] = list(state["messages"])
    last = msgs[-1]
    if getattr(last, "tool_calls", None):
        msgs.append(
            SystemMessage(
                "Search budget reached or tools are no longer available for this step. "
                "Produce the structured plan from the conversation so far, or ask for user clarification."
            )
        )

    finalize = SystemMessage(
        "Emit the final structured plan. If the planner asked the user a clarifying question in plain text, "
        "set additional_input_required=True and clarification_question. Otherwise set additional_input_required=False and "
        "provide plan items: query (Wikipedia / follow-up), optional url (https from search results when used), reason."
    )
    planner_output: PlannerOutput = planner_llm_with_output.invoke([finalize] + msgs)

    if planner_output.additional_input_required:
        q = planner_output.clarification_question or (
            planner_output.plan[0].query if planner_output.plan else "Please provide more detail."
        )
        return {
            "messages": [AIMessage(content=f"Planner needs clarification: {q}")],
            "planning_complete": False,
            "user_input_needed": True,
        }

    plan = planner_output.plan or []
    if not plan:
        return {
            "messages": [AIMessage(content="Planner produced an empty plan.")],
            "planning_complete": False,
            "user_input_needed": True,
        }

    used_search = any(
        isinstance(m, ToolMessage) and getattr(m, "name", "") == "google_search"
        for m in state["messages"]
    )
    plan_chunks: List[str] = []
    for item in plan:
        u = item.url or "none"
        plan_chunks.append(
            f"- query: {item.query}\n  reason: {item.reason}\n  url: {u}"
        )
    tag = "(google_search used in planner)" if used_search else "(no planner web search)"
    plan_text = f"Planning complete {tag}. Items:\n" + "\n".join(plan_chunks)
    return {
        "messages": [AIMessage(content=plan_text)],
        "planning_complete": True,
        "user_input_needed": False,
        "success_criteria": "Deliver an accurate, well-grounded answer using the plan (Wikipedia and/or fetched URLs as appropriate).",
    }


In [ ]:
def route_planner_output(state: State) -> str:
    if state.get("user_input_needed"):
        return END
    if state.get("planning_complete"):
        return "worker"
    return END

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    criteria = state.get("success_criteria") or "Satisfy the user's request using the plan."
    system_message = f"""You are a research worker. You have Wikipedia search and fetch_web_page (for URLs in the plan).
Use wiki search for query terms; use fetch_web_page when the plan includes a concrete http(s) URL to read.
When you have enough evidence, write the FINAL answer as plain text or markdown in your last message (no tool calls).

The final message is emailed to the user: make it a detailed, self-contained report. Include:
- A clear title line or opening sentence stating what you investigated.
- Organized sections (e.g. summary, key findings, supporting detail) as appropriate.
- Citations or mentions of sources (which articles/pages you used).
- Nuance, caveats, or limits of the evidence when relevant.
Aim for thoroughness over brevity while staying accurate and grounded in retrieved content.

Success criteria: {criteria}
"""
    if state.get("feedback_on_work"):
        system_message += f"""
Previous attempt was rejected. Address this feedback:
{state['feedback_on_work']}
"""

    messages = [SystemMessage(content=system_message)] + state["messages"]
    response = worker_llm_with_tools.invoke(messages)

    out: Dict[str, Any] = {"messages": [response]}
    if not getattr(response, "tool_calls", None):
        out["last_report"] = response.content or ""
    return out

In [ ]:
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return "evaluator"

In [ ]:
def evaluator(state: State) -> State:
    last = state["messages"][-1]
    last_response = getattr(last, "content", None) or str(last)

    system_message = """You are an evaluator. Decide if the worker's last answer meets the success criteria,
and whether user input is still needed."""

    user_message = f"""Conversation:
{format_conversation(state['messages'])}

Success criteria:
{state.get('success_criteria', '')}

Worker response to judge:
{last_response}
"""
    if state.get("feedback_on_work"):
        user_message += f"\nPrior feedback (avoid repeat failures): {state['feedback_on_work']}\n"

    eval_result = evaluator_llm_with_output.invoke(
        [SystemMessage(content=system_message), HumanMessage(content=user_message)]
    )
    return {
        "messages": [AIMessage(content=f"Evaluator: {eval_result.feedback}")],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed,
    }

In [ ]:
def route_based_on_evaluation(state: State) -> str:
    if state.get("success_criteria_met"):
        return "notifier"
    if state.get("user_input_needed"):
        return END
    return "worker"

In [ ]:
def guardrail_node(state: State) -> Dict[str, Any]:
    text = ""
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            text = str(m.content)
            break
    try:
        input_guardrail(text)
        return {"guard_ok": True}
    except ValueError as e:
        return {
            "guard_ok": False,
            "messages": [AIMessage(content=f"Input blocked: {e}")],
        }


def route_after_guard(state: State) -> str:
    return "planner_llm" if state.get("guard_ok") else END


In [ ]:
def notifier(state: State) -> Dict[str, Any]:
    """Runs only after evaluator success: one push (single invoke each; no tool loop)."""
    report = (state.get("last_report") or "").strip() or "(empty report)"
    summary = (report[:450] + "…") if len(report) > 450 else report
    parts: List[str] = []

    push_result = send_push_notification.invoke({"message": summary})
    parts.append(f"push: {push_result}")

    return {"messages": [AIMessage(content="Notifier: " + " | ".join(parts))]}

### Build Graph

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("guardrail", guardrail_node)
graph_builder.add_node("planner_llm", planner_llm)
graph_builder.add_node("planner_search", planner_search)
graph_builder.add_node("planner_finalize", planner_finalize)
graph_builder.add_node("worker", worker)
graph_builder.add_node("evaluator", evaluator)
graph_builder.add_node("notifier", notifier)
graph_builder.add_node("tools", worker_tool_node)

graph_builder.add_edge(START, "guardrail")
graph_builder.add_conditional_edges(
    "guardrail", route_after_guard, {END: END, "planner_llm": "planner_llm"}
)
graph_builder.add_conditional_edges(
    "planner_llm",
    route_planner_tools,
    {"planner_search": "planner_search", "planner_finalize": "planner_finalize"},
)
graph_builder.add_edge("planner_search", "planner_llm")
graph_builder.add_conditional_edges(
    "planner_finalize", route_planner_output, {END: END, "worker": "worker"}
)
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges(
    "evaluator",
    route_based_on_evaluation,
    {END: END, "notifier": "notifier", "worker": "worker"},
)
graph_builder.add_edge("notifier", END)

graph = graph_builder.compile(checkpointer=memory)

In [ ]:
#display grapgh workflow uncomment to display graph
display(Image(graph.get_graph().draw_mermaid_png()))

#### Main chat functions

In [ ]:
def _chat_preview(messages: List[Any]) -> str:
    """displays Only the latest assistant text."""
    for m in reversed(messages):
        if isinstance(m, AIMessage):
            c = (m.content or "").strip()
            if c:
                return c
    return "Working…"


async def stream_chat(message: str, history: list, thread_id: str):
    if not message or not str(message).strip():
        yield history, thread_id
        return

    msg = str(message).strip()
    hist = list(history)
    hist.append({"role": "user", "content": msg})
    hist.append({"role": "assistant", "content": "Running workflow…"})
    yield hist, thread_id

    config = {"configurable": {"thread_id": thread_id}}
    state_in = {"messages": [HumanMessage(content=msg)]}

    try:
        async for state in graph.astream(state_in, config, stream_mode="values"):
            msgs = state.get("messages") or []
            hist[-1] = {"role": "assistant", "content": _chat_preview(msgs)}
            yield hist, thread_id
    except Exception as e:
        hist[-1] = {"role": "assistant", "content": f"**Error:** {e}"}
        yield hist, thread_id


def new_conversation():
    return [], str(uuid.uuid4())

#### Gradio UI

In [ ]:
with gr.Blocks(title="Information Companion") as ui:
    gr.Markdown("# Information Companion\nStreaming run (updates as each graph step completes). Use **New conversation** for a fresh thread.")
    thread_id = gr.State(lambda: str(uuid.uuid4()))
    chat = gr.Chatbot(type="messages", height=480, label="Chat")
    box = gr.Textbox(placeholder="Ask something (10+ characters)…", label="Your message", lines=2)
    with gr.Row():
        go = gr.Button("Send", variant="primary")
        reset = gr.Button("New conversation")

    go.click(stream_chat, [box, chat, thread_id], [chat, thread_id]).then(
        lambda: "", outputs=box
    )
    box.submit(stream_chat, [box, chat, thread_id], [chat, thread_id]).then(
        lambda: "", outputs=box
    )
    reset.click(new_conversation, outputs=[chat, thread_id])

ui.launch(inbrowser=True)